# Download IG price history

Pulls **OHLC candles** for an IG market from the IG REST API, then can save
them to a timestamped
`data/ig_<epic-slug>_<resolution-suffix>_<timestamp>.csv` and/or upsert them
into the resolution-specific `candles_<suffix>` table (e.g. `candles_1d`) of
`data/ig_market_data.db` (same schema the sibling `03_view_ig_prices.ipynb`
reads) — each output is independently toggled via `SAVE_CSV` / `SAVE_DB`.

**The download window and resolution are set in this notebook**, in section 1:
`START`, `END`, `RESOLUTION`. Credentials, the `EPIC`, and the `SAVE_CSV` /
`SAVE_DB` toggles still come from a `.env` file in this folder (git-ignored):

| var | meaning |
| --- | --- |
| `IG_API_KEY` | IG API key for the environment set by `IG_ACCOUNT_TYPE` |
| `IG_USERNAME` | IG login |
| `IG_PASSWORD` | IG password |
| `IG_ACCOUNT_TYPE` | `demo` or `live` (default `demo`) |
| `IG_EPIC` | market to pull candles for (default `IX.D.NASDAQ.IFA.IP`) |
| `IG_SAVE_CSV` | write the timestamped CSV (`true`/`false`, default `true`) |
| `IG_SAVE_DB` | upsert into the SQLite DB (`true`/`false`, default `false`) |

`IG_RESOLUTION` / `IG_DAYS_BACK` in `.env` are **not** used here (the other
notebooks still read `IG_RESOLUTION`) — set `RESOLUTION` / `START` / `END`
below instead.

The download needs `requests` + `python-dotenv`. `pandas` / `matplotlib` are
used for the optional preview and chart at the end.

## 1. Load credentials + set the download window

`Config.from_env()` (in `igmarket/config.py`) loads `.env` for the IG
credentials, `EPIC`, and the `SAVE_CSV` / `SAVE_DB` toggles. The candle
`RESOLUTION` and the `START` / `END` of the window are set right here as
notebook variables — edit them and re-run.

In [1]:
# One-time setup: install this repo's `igmarket` package (editable) + deps.
# Re-running is cheap (-q, and pip no-ops if it's already satisfied).
import subprocess, sys
from pathlib import Path

_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{_root}[dev]"],
    check=True,
)

CompletedProcess(args=['C:\\Users\\justi\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', '-m', 'pip', 'install', '-q', '-e', 'C:\\repos\\github.com\\jupyter-notebooks[dev]'], returncode=0)

In [ ]:
# --- edit these: download window (UTC) + candle resolution ------------------
START = "2024-01-01"   # inclusive. "YYYY-MM-DD" or "YYYY-MM-DDTHH:MM:SS"
END = None             # None -> now
RESOLUTION = "DAY"     # MINUTE, MINUTE_5, MINUTE_10, HOUR, DAY, WEEK, ...
# --------------------------------------------------------------------------

import sqlite3
from datetime import datetime, timezone

from igmarket.config import Config

cfg = Config.from_env()  # credentials, EPIC, SAVE_CSV / SAVE_DB, data paths

start_dt = datetime.fromisoformat(START).replace(tzinfo=timezone.utc)
end_dt = (
    datetime.now(timezone.utc)
    if END is None
    else datetime.fromisoformat(END).replace(tzinfo=timezone.utc)
)

CSV_PATH = cfg.csv_path_for(RESOLUTION)  # fresh timestamped path for this resolution
DB_PATH = cfg.db_path

print(f"Loaded config from {cfg.env_path.resolve()}")
print(f"user={cfg.username!r}  account_type={cfg.account_type!r}  api_key=***{cfg.api_key[-4:]}")
print(f"EPIC={cfg.epic!r}  RESOLUTION={RESOLUTION!r}")
print(f"window (UTC): {start_dt:%Y-%m-%d %H:%M}  ->  {end_dt:%Y-%m-%d %H:%M}")
print(f"SAVE_CSV={cfg.save_csv}  SAVE_DB={cfg.save_db}")

## 2. Download

Opens an authenticated `IGSession` (`igmarket/ig_session.py` — `POST /session`
to authenticate, then `GET /prices/{epic}` (`Version: 3`) with
`resolution`/`from`/`to`/`pageSize`/`pageNumber` to page through history),
then fetches `RESOLUTION` candles for `cfg.epic` between `START` and `END`
via `get_prices()`. Prints the candle count, the UTC time range covered, and
the historical-data allowance remaining on the account — the raw
`metadata.allowance` field names (`remainingAllowance`, `totalAllowance`,
`allowanceExpiry`) live in `constants/allowance_fields.py` (`AllowanceField`).

In [ ]:
from igmarket.constants.allowance_fields import AllowanceField
from igmarket.constants.ig_candle_fields import CandleField
from igmarket.ig_session import IGSession

ig = IGSession(cfg.api_key, cfg.username, cfg.password, cfg.account_type)

fmt = "%Y-%m-%dT%H:%M:%S"
raw_candles, allowance = ig.get_prices(
    cfg.epic, RESOLUTION, start_dt.strftime(fmt), end_dt.strftime(fmt)
)

print(f"{len(raw_candles)} {RESOLUTION} candles for {cfg.epic}")
if raw_candles:
    print(f"  range (UTC): {raw_candles[0][CandleField.SNAPSHOT_TIME_UTC]}  ->  {raw_candles[-1][CandleField.SNAPSHOT_TIME_UTC]}")
if allowance:
    print(
        f"  historical-data allowance: {allowance.get(AllowanceField.REMAINING_ALLOWANCE)}"
        f" / {allowance.get(AllowanceField.TOTAL_ALLOWANCE)} remaining"
        f" (resets in {allowance.get(AllowanceField.ALLOWANCE_EXPIRY)}s)"
    )

## 3. Persist

Two outputs, each independently toggled in the config cell — `SAVE_CSV` and
`SAVE_DB` — so a run can write either, both, or neither.

### CSV — full bid/ask/mid OHLC

Toggled by `SAVE_CSV`. Row-flattening lives in `candle_csv.py`
(`candle_to_row()`); the raw IG field-name constants (`CandleField`,
`PriceField`) live in `constants/ig_candle_fields.py`, and column headers in
`constants/csv_headers.py` (`CSV_HEADERS`) — imported by the cells below so
they can't drift out of sync. Columns:

- `snapshot_time_utc` — IG's `snapshotTimeUTC`, ISO `YYYY-MM-DDTHH:MM:SS`, UTC
  (not the exchange-local `snapshotTime`)
- `{open,high,low,close}_{bid,ask,mid}_price` — 12 price columns, raw IG
  values; `mid` is derived (`(bid + ask) / 2`), not requested separately
- `last_traded_volume`

Each run writes a **new, timestamped file**
(`data/ig_<epic-slug>_<resolution-suffix>_<RUN_TIMESTAMP>.csv` — `_epic_slug()`
in `config.py` takes the instrument segment of the dot-separated `EPIC`, e.g.
`IX.D.NASDAQ.IFA.IP` -> `nasdaq`, and `RESOLUTION` maps to a filename-friendly
suffix via `constants/csv_filename_suffix.py`'s `RESOLUTION_CSV_SUFFIX`, e.g.
`DAY` -> `daily`, `MINUTE_10` -> `10min`) rather than overwriting a fixed
path, so nothing is ever lost between runs — but successive runs' CSVs
accumulate in `data/` and are never cleaned up automatically.

In [ ]:
# --- CSV (full bid/ask/mid OHLC) ----------------------------------------------
def write_csv():
    if not cfg.save_csv:
        print("SAVE_CSV is False - skipping CSV write")
        return

    import csv

    from igmarket.candle_csv import candle_to_row
    from igmarket.constants.csv_headers import CSV_HEADERS

    with CSV_PATH.open("w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(CSV_HEADERS)
        for c in raw_candles:
            writer.writerow(candle_to_row(c))
    print(f"wrote {CSV_PATH.resolve()}  ({len(raw_candles)} rows)")


write_csv()

### SQLite — upsert into `candles_<suffix>`

Toggled by `SAVE_DB`.

- **Target:** a resolution-specific table — `candles_1d` for `DAY`,
  `candles_10min` for `MINUTE_10`, etc. — in `data/ig_market_data.db`. Table
  creation and the `resolution` -> table-name mapping live in `candle_db.py`
  (`init_candles_table()`, `table_name_for_resolution()`), backed by
  `constants/resolutions.py` (`RESOLUTION_TABLE_SUFFIX`), so
  `03_view_ig_prices.ipynb` derives the same table name from a resolution.
- **Columns:** `epic`, then the flattened price columns matching the CSV
  output today — `constants/db_headers.py`'s `DB_HEADERS`, a copy of
  `constants/csv_headers.py`'s `CSV_HEADERS` kept as its own list so the two
  schemas can diverge independently — built with the same
  `candle_csv.candle_to_row()` used for the CSV. The resolution isn't stored
  — it's fixed per table and recoverable from the table name.
- **Key:** `snapshot_time_utc` (UTC, from IG's `snapshotTimeUTC` — not the
  exchange-local `snapshotTime`); `UNIQUE(epic, snapshot_time_utc)` per table
  (resolution no longer needs to be part of the key — it's fixed per table).
- **Write mode:** `INSERT OR IGNORE`. Existing rows are **never updated** —
  re-downloading a still-forming candle keeps the stale row already in
  the DB.

In [ ]:
# --- SQLite (flattened prices, same schema as 03_view_ig_prices.ipynb) --------------------
def write_db():
    if not cfg.save_db:
        print("SAVE_DB is False - skipping SQLite upsert")
        return

    from igmarket.candle_csv import candle_to_row
    from igmarket.candle_db import INSERT_COLUMNS, init_candles_table
    from igmarket.constants.ig_candle_fields import CandleField

    conn = sqlite3.connect(DB_PATH)
    table = init_candles_table(conn, RESOLUTION)
    rows = [
        (cfg.epic, *candle_to_row(c))
        for c in raw_candles
        if c.get(CandleField.SNAPSHOT_TIME_UTC)
    ]
    before = conn.total_changes
    placeholders = ", ".join("?" for _ in INSERT_COLUMNS)
    conn.executemany(
        f"INSERT OR IGNORE INTO {table} ({', '.join(INSERT_COLUMNS)}) "
        f"VALUES ({placeholders})",
        rows,
    )
    conn.commit()
    inserted = conn.total_changes - before
    print(f"{DB_PATH.resolve()}: +{inserted} new row(s) in `{table}` "
          f"({len(rows) - inserted} already present)")
    conn.close()


write_db()

## 4. preview + close (mid) chart

Needs `pandas` and `matplotlib` (`%pip install pandas matplotlib`). Builds a
frame straight from the candles just downloaded (not from disk), so it works
whether or not `SAVE_CSV` / `SAVE_DB` wrote anything this run: a tail-10 table
+ `describe()`, then a close (mid) line chart with a shaded high/low band.

In [ ]:
try:
    import pandas as pd
except ModuleNotFoundError:
    print("pandas not installed - run:  %pip install pandas")
    df = None
else:
    from igmarket.candle_csv import candle_to_row
    from igmarket.constants.csv_headers import CSV_HEADERS

    # Straight from the candles just downloaded, so the preview and chart work
    # even when SAVE_CSV and SAVE_DB are both False (nothing written to disk).
    df = pd.DataFrame((candle_to_row(c) for c in raw_candles), columns=CSV_HEADERS)
    df["snapshot_time_utc"] = pd.to_datetime(df["snapshot_time_utc"])
    df = df.set_index("snapshot_time_utc").sort_index()
    if df.empty:
        print(f"Download returned 0 {RESOLUTION} candles - nothing to preview.")
    else:
        display(df.tail(10))
        display(df.describe())

In [ ]:
if df is None:
    print("Run the pandas cell above first.")
elif df.empty:
    print(
        f"No {RESOLUTION} candles between {START} and {END or 'now'} - "
        f"nothing to plot (e.g. a weekend / market-closed window); widen START/END."
    )
else:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        print("matplotlib not installed - run:  %pip install matplotlib")
    else:
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(df.index, df["close_mid_price"], color="#26a69a")
        ax.fill_between(df.index, df["low_mid_price"], df["high_mid_price"], color="#26a69a", alpha=0.15)
        ax.set_title(f"{cfg.epic}  {RESOLUTION}  close (mid)  -  {START} to {END or 'now'}")
        ax.set_ylabel("price (mid)")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()